In [1]:
from cosim import Flow, Manager, Kio
import cosim.db as db
import requests
import os
os.makedirs("__infra__", exist_ok=True)

# Task Offloading

* Load the infra

In [2]:
infra = Kio.LoadJSON('__infra__.json')
infra

{'ue': {'name': 'ue',
  'nhost': '127.0.0.1',
  'nport': '9800',
  'dhost': '127.0.0.1',
  'dport': '9801',
  'https': False,
  'xy': [0.0, 0.0]},
 'edge': {'name': 'edge',
  'nhost': '127.0.0.1',
  'nport': '9810',
  'dhost': '127.0.0.1',
  'dport': '9811',
  'https': False,
  'xy': [0.0, 0.0]},
 'cloud': {'name': 'cloud',
  'nhost': '127.0.0.1',
  'nport': '9820',
  'dhost': '127.0.0.1',
  'dport': '9821',
  'https': False,
  'xy': [0.0, 0.0]}}

* Next, Prepare the flow for offloading

In [5]:
node_id = "ue"
offloader = "http://127.0.0.1:9696"

custom_input_x = 10
custom_input_file_name = 'custom_x'
# create an input file at the desired location - should be pickle file
Kio.SavePICK(f'__infra__/ue/data/{custom_input_file_name}', custom_input_x)

response = requests.post(
    f'{offloader}/new',
    json= dict(
            node_id = node_id,
            flow_name = 'example',
            input = custom_input_file_name, 
        )
)

resd = response.json()
resd

{'decision': {'E': 'ue', 'I1': 'edge', 'I2': 'edge', 'X': 'cloud'},
 'nodeid': 'ue',
 'offloading_status': {'E': [200,
   '{"received":"ue_20260118095002195945_E"}\n',
   'http://127.0.0.1:9800/add'],
  'I1': [200,
   '{"received":"ue_20260118095002195945_I1"}\n',
   'http://127.0.0.1:9810/add'],
  'I2': [200,
   '{"received":"ue_20260118095002195945_I2"}\n',
   'http://127.0.0.1:9810/add'],
  'X': [200,
   '{"received":"ue_20260118095002195945_X"}\n',
   'http://127.0.0.1:9820/add']},
 'output_filename': 'ue_20260118095002195945_X_o',
 'rstatus': [200,
  '{"data":["x"],"received":"example/ue_20260118095002195945_E"}\n',
  'http://127.0.0.1:9800/notify']}

In [6]:
Kio.LoadPICK(f'__infra__/ue/data/{resd["output_filename"]}')

15.83772233983162

# End

* cleanup temporary files
```shell
rm -rf __infra__ && rm -f __infra__.json && rm -f __policy*__.py && pyclean -v .
```

---